# MedSegDiff — Thigh Muscle Segmentation Training (Lambda)

Trains one DDPM-based binary segmentation model per muscle on the myosegmenTUM
Dixon water images, then runs inference on both **water** and **fat-fraction** stacks.

Architecture: 4-level U-Net conditioned on water (+optionally fat-fraction)
concatenated with the noisy mask channel.  DDIM sampling at inference (50 steps).

## Before running — upload to Lambda

```bash
# myosegmenTUM dataset
rsync -avz -e "ssh -i /tmp/lambda_key -o StrictHostKeyChecking=no" \
  /tmp/docker-desktop-root/run/desktop/mnt/host/c/Projects/dissector/eval_notebooks/myosegmenTUM \
  your machine:~/

# medsegdiff package
rsync -avz -e "ssh -i /tmp/lambda_key -o StrictHostKeyChecking=no" \
  /tmp/docker-desktop-root/run/desktop/mnt/host/c/Projects/dissector/medsegdiff/ \
  your machine:~/medsegdiff/
```

## Download results when done

```bash
# checkpoints
rsync -avz -e "ssh -i /tmp/lambda_key -o StrictHostKeyChecking=no" \
  your machine:~/medsegdiff_ckpts/ \
  /tmp/docker-desktop-root/run/desktop/mnt/host/c/Projects/dissector/eval_notebooks/medsegdiff_ckpts/

# water segmentations
rsync -avz -e "ssh -i /tmp/lambda_key -o StrictHostKeyChecking=no" \
  your machine:~/medsegdiff_segs_water/ \
  /tmp/docker-desktop-root/run/desktop/mnt/host/c/Projects/dissector/eval_notebooks/medsegdiff_segs_water/

# fat-fraction segmentations
rsync -avz --mkpath -e "ssh -i /tmp/lambda_key -o StrictHostKeyChecking=no" \
  your machine:~/medsegdiff_segs_fatfrac/ \
  /tmp/docker-desktop-root/run/desktop/mnt/host/c/Projects/dissector/eval_notebooks/medsegdiff_segs_fatfrac/
```

**Terminate the instance when done.**

In [ ]:
import subprocess, sys

# Only install if not already present — avoids slow pip runs on reconnect
def _ensure(*pkgs):
    import importlib
    missing = [p for p in pkgs if importlib.util.find_spec(p.split('[')[0].replace('-','_')) is None]
    if missing:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q'] + list(missing))
    else:
        print('Packages already installed:', ', '.join(pkgs))

_ensure('SimpleITK', 'tqdm', 'torchvision')

import torch
print('PyTorch :', torch.__version__)
print('CUDA    :', torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

In [ ]:
import os, sys

MEDSEGDIFF_DIR = os.path.expanduser('~/medsegdiff')
GT_BASE        = os.path.expanduser('~/myosegmenTUM')
CKPT_DIR       = os.path.expanduser('~/medsegdiff_ckpts')
SEG_DIR        = os.path.expanduser('~/medsegdiff_segs_water')
SEG_DIR_FF     = os.path.expanduser('~/medsegdiff_segs_fatfrac')

for path, label in [
    (MEDSEGDIFF_DIR, 'medsegdiff package'),
    (GT_BASE,        'myosegmenTUM data'),
]:
    exists = os.path.isdir(path)
    print(f'{"OK" if exists else "MISSING"}: {label} ({path})')
    if not exists:
        raise FileNotFoundError(f'Upload {label} first — see rsync commands above')

os.makedirs(CKPT_DIR,   exist_ok=True)
os.makedirs(SEG_DIR,    exist_ok=True)
os.makedirs(SEG_DIR_FF, exist_ok=True)

if MEDSEGDIFF_DIR not in sys.path:
    sys.path.insert(0, MEDSEGDIFF_DIR)

from dataset   import DixonThighDataset, GT_LABELS, discover_subjects, train_val_split
from unet      import UNet
from diffusion import GaussianDiffusion
print('Imports OK.  Muscles:', list(GT_LABELS))

In [ ]:
# ── Training configuration — edit here ───────────────────────────────────────
MUSCLES     = list(GT_LABELS)   # train all 4; or e.g. ['R_gracilis']
IMG_SIZE    = 256
BASE_CH     = 64
T_DIM       = 256
DROPOUT     = 0.1
T_STEPS     = 1000
EPOCHS      = 300
BATCH_SIZE  = 8
LR          = 1e-4
USE_FF      = True    # include fat-fraction as second image channel
VAL_FRAC    = 0.2
SAVE_EVERY  = 25      # checkpoint frequency (epochs)
VAL_EVERY   = 25      # validation Dice frequency (expensive; runs DDIM)
DDIM_STEPS  = 50
NUM_WORKERS = 4

# Set True to skip any muscle whose full training run already completed
# (i.e. a checkpoint exists at epoch == EPOCHS). Useful after reconnecting.
SKIP_TRAINED = True

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)

all_subjects = discover_subjects(GT_BASE)
train_subj, val_subj = train_val_split(all_subjects, val_fraction=VAL_FRAC)
print(f'Subjects: {len(train_subj)} train / {len(val_subj)} val')
print('Train:', train_subj[:5], '...')
print('Val  :', val_subj)

In [ ]:
import time
from torch.utils.data import DataLoader
import torch.optim as optim


def validate(model, diffusion, loader):
    model.eval()
    scores = []
    with torch.no_grad():
        for img, mask in loader:
            img, mask = img.to(DEVICE), mask.to(DEVICE)
            pred = diffusion.ddim_sample(model, img, num_steps=DDIM_STEPS)
            scores.append(diffusion.dice((pred > 0).float(), (mask > 0).float()))
    model.train()
    return float(sum(scores) / max(len(scores), 1))


def train_muscle(muscle):
    print(f'\n{"="*60}')
    print(f' Training: {muscle}')
    print(f'{"="*60}')

    train_ds = DixonThighDataset(
        GT_BASE, muscle, train_subj,
        img_size=IMG_SIZE, use_ff=USE_FF, augment=True,
    )
    val_ds = DixonThighDataset(
        GT_BASE, muscle, val_subj,
        img_size=IMG_SIZE, use_ff=USE_FF, augment=False,
    )
    print(f'Slices: {len(train_ds)} train / {len(val_ds)} val')

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                              num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
    val_loader   = DataLoader(val_ds,   batch_size=4, shuffle=False,
                              num_workers=NUM_WORKERS, pin_memory=True)

    img_ch = train_ds.n_image_channels
    model  = UNet(img_ch=img_ch, base=BASE_CH, t_dim=T_DIM, dropout=DROPOUT).to(DEVICE)
    diff   = GaussianDiffusion(T=T_STEPS, device=DEVICE)
    opt    = optim.AdamW(model.parameters(), lr=LR)
    sched  = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)

    print(f'Parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')

    ckpt_path = os.path.join(CKPT_DIR, f'{muscle}_latest.pt')
    best_path = os.path.join(CKPT_DIR, f'{muscle}_best.pt')
    start_epoch, best_dice = 0, 0.0

    if os.path.exists(ckpt_path):
        ckpt = torch.load(ckpt_path, map_location=DEVICE)
        model.load_state_dict(ckpt['model'])
        opt.load_state_dict(ckpt['optimizer'])
        sched.load_state_dict(ckpt['scheduler'])
        start_epoch = ckpt['epoch'] + 1
        best_dice   = ckpt.get('best_dice', 0.0)
        print(f'Resumed from epoch {start_epoch} (best Dice {best_dice:.4f})')

    log_path = os.path.join(CKPT_DIR, f'{muscle}_log.csv')
    if start_epoch == 0:
        with open(log_path, 'w') as f:
            f.write('epoch,loss,val_dice,lr\n')

    model.train()
    t0 = time.time()

    for epoch in range(start_epoch, EPOCHS):
        total_loss = 0.0
        for img, mask in train_loader:
            img, mask = img.to(DEVICE), mask.to(DEVICE)
            t = torch.randint(0, T_STEPS, (img.shape[0],), device=DEVICE)
            loss = diff.training_loss(model, mask, img, t)
            opt.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            total_loss += loss.item()

        sched.step()
        avg_loss = total_loss / len(train_loader)
        lr_now   = sched.get_last_lr()[0]

        val_dice = 0.0
        if len(val_ds) > 0 and (epoch + 1) % VAL_EVERY == 0:
            val_dice = validate(model, diff, val_loader)
            if val_dice > best_dice:
                best_dice = val_dice
                torch.save({'model': model.state_dict(), 'epoch': epoch,
                            'best_dice': best_dice}, best_path)
                print(f'  *** Best Dice {best_dice:.4f} — saved')

        elapsed = time.time() - t0
        if (epoch + 1) % 10 == 0 or epoch == start_epoch:
            print(f'  [{muscle}] Epoch {epoch+1:4d}/{EPOCHS}  '
                  f'loss={avg_loss:.5f}  val_dice={val_dice:.4f}  '
                  f'lr={lr_now:.2e}  {elapsed:.0f}s')

        with open(log_path, 'a') as f:
            f.write(f'{epoch+1},{avg_loss:.6f},{val_dice:.4f},{lr_now:.2e}\n')

        if (epoch + 1) % SAVE_EVERY == 0 or epoch + 1 == EPOCHS:
            torch.save({'model': model.state_dict(), 'optimizer': opt.state_dict(),
                        'scheduler': sched.state_dict(), 'epoch': epoch,
                        'best_dice': best_dice}, ckpt_path)

    print(f'  {muscle} done. Best val Dice: {best_dice:.4f}  ({time.time()-t0:.0f}s total)')
    return model, diff


print('Training helpers defined.')

In [ ]:
# ── Run training for all muscles ─────────────────────────────────────────────
trained = {}
for muscle in MUSCLES:
    ckpt_path = os.path.join(CKPT_DIR, f'{muscle}_latest.pt')
    if SKIP_TRAINED and os.path.exists(ckpt_path):
        ckpt = torch.load(ckpt_path, map_location='cpu')
        if ckpt.get('epoch', -1) + 1 >= EPOCHS:
            print(f'[skip] {muscle} — already completed epoch {EPOCHS}')
            # Re-load into memory so inference cell can use it
            from unet      import UNet
            from diffusion import GaussianDiffusion
            from dataset   import DixonThighDataset
            tmp_ds = DixonThighDataset(GT_BASE, muscle, train_subj[:1],
                                       img_size=IMG_SIZE, use_ff=USE_FF)
            model = UNet(img_ch=tmp_ds.n_image_channels, base=BASE_CH,
                         t_dim=T_DIM, dropout=DROPOUT).to(DEVICE)
            model.load_state_dict(ckpt['model'])
            model.eval()
            diff = GaussianDiffusion(T=T_STEPS, device=DEVICE)
            trained[muscle] = (model, diff)
            continue

    model, diff = train_muscle(muscle)
    trained[muscle] = (model, diff)

print('\nAll muscles trained.')

## Running offline-resilient (recommended for long jobs)

Jupyter cells are killed if the SSH/browser connection drops or times out.
Use **tmux** to run training in a persistent terminal session that survives disconnects.

### 1 — Export this notebook to a standalone script (run once)
```bash
jupyter nbconvert --to script lambda_medsegdiff.ipynb --output ~/train_medsegdiff
```

### 2 — Launch in a tmux session
```bash
tmux new-session -d -s msd 'python ~/train_medsegdiff.py 2>&1 | tee ~/msd_log.txt'
```

### 3 — Reconnect / check progress (after re-SSHing)
```bash
tmux attach -t msd          # re-attach to the live session
# or just tail the log:
tail -f ~/msd_log.txt
```

### 4 — Kill the session when done
```bash
tmux kill-session -t msd
```

**Checkpoints** are saved every `SAVE_EVERY` epochs to `~/medsegdiff_ckpts/`.  
If the job is interrupted, re-running will automatically resume from the last checkpoint.  
Set `SKIP_TRAINED = True` (already default) to skip any muscle that reached epoch `EPOCHS`.

In [ ]:
# ── Inference — segment water AND fat-fraction stacks ────────────────────────
# The model was trained with [water, fat-fraction] as 2-channel input (USE_FF=True).
# Both modality runs use the same channel ordering and the same trained weights;
# only the output filename and directory change so the evaluation pipeline can
# pick up results for each modality independently.

import glob as _glob, re as _re
import torch.nn.functional as F
import numpy as np
import SimpleITK as sitk
from dataset import _norm


@torch.no_grad()
def segment_stack(model, diff, wpath, ff_path, img_ch):
    w_arr = sitk.GetArrayFromImage(sitk.ReadImage(wpath)).astype('float32')
    D, H, W = w_arr.shape
    ff_arr = sitk.GetArrayFromImage(sitk.ReadImage(ff_path)).astype('float32') \
             if ff_path and os.path.exists(ff_path) else None

    pred_vol = np.zeros((D, H, W), dtype=np.uint8)
    model.eval()
    for sl in range(D):
        channels = [torch.from_numpy(_norm(w_arr[sl])).unsqueeze(0)]
        if ff_arr is not None and img_ch == 2:
            channels.append(torch.from_numpy(_norm(ff_arr[sl])).unsqueeze(0))
        img_t  = torch.cat(channels, dim=0) * 2.0 - 1.0
        img_r  = F.interpolate(img_t.unsqueeze(0).to(DEVICE),
                               size=(IMG_SIZE, IMG_SIZE),
                               mode='bilinear', align_corners=False)
        pred   = diff.ddim_sample(model, img_r, num_steps=DDIM_STEPS)
        pred_r = F.interpolate(pred, size=(H, W), mode='bilinear', align_corners=False)
        pred_vol[sl] = (pred_r.squeeze().cpu().numpy() > 0).astype(np.uint8)
    return pred_vol


RUNS = [
    # (primary_modality, secondary_modality, output_dir)
    ('WATER',       'FATFRACTION', SEG_DIR),
    ('FATFRACTION', 'WATER',       SEG_DIR_FF),
]

img_ch = 2 if USE_FF else 1

for primary_mod, secondary_mod, out_dir in RUNS:
    print(f'\n{"="*60}')
    print(f' Inference: {primary_mod} stacks  ->  {out_dir}')
    print(f'{"="*60}')

    for subj in sorted(discover_subjects(GT_BASE)):
        subj_dir   = os.path.join(GT_BASE, subj)
        prim_files = sorted(_glob.glob(
            os.path.join(subj_dir, 'ImageData',
                         f'{subj}_{primary_mod}',
                         f'{subj}_{primary_mod}_stack*.nii')
        ))

        for prim_path in prim_files:
            m = _re.search(r'stack(\d+)\.nii$', prim_path)
            if not m:
                continue
            stack   = m.group(1)
            stem    = f'{subj}_{primary_mod}_stack{stack}'
            out_npz = os.path.join(out_dir, f'{stem}_medsegdiff.npz')

            if os.path.exists(out_npz):
                print(f'  skip (done): {stem}')
                continue

            sec_path = os.path.join(subj_dir, 'ImageData',
                                    f'{subj}_{secondary_mod}',
                                    f'{subj}_{secondary_mod}_stack{stack}.nii')

            # Always pass water as ch0, fat-fraction as ch1 (matches training)
            if primary_mod == 'WATER':
                wpath, ff_path = prim_path, sec_path
            else:
                wpath, ff_path = sec_path, prim_path

            if not os.path.exists(wpath):
                print(f'  skip (no water): {stem}')
                continue

            print(f'  {stem} ...', end=' ', flush=True)
            all_masks = {}
            for muscle, (model, diff) in trained.items():
                all_masks[muscle] = segment_stack(model, diff, wpath, ff_path, img_ch)
            np.savez_compressed(out_npz, **all_masks)
            total = sum(v.sum() for v in all_masks.values())
            print(f'{total:,} voxels  -> {out_npz}')

print('\nInference complete.')

In [ ]:
# Sanity check
for label, seg_dir in [('water', SEG_DIR), ('fat fraction', SEG_DIR_FF)]:
    results = sorted(_glob.glob(os.path.join(seg_dir, '*.npz')))
    print(f'\n{label}: {len(results)} NPZ files in {seg_dir}')
    if results:
        sample = np.load(results[0])
        print(f'  Sample: {results[0]}')
        for k in sample.files:
            arr = sample[k]
            print(f'    {k}: shape={arr.shape}  voxels={int(arr.sum()):,}')